In [1]:
#imports and proxies
import json
import pyodbc
from simple_salesforce import Salesforce
import pandas as pd
import numpy as np
pd.set_option("display.max_colwidth", None)

#set proxies
proxies = {
 	"http": 'http://web.prod.proxy.cargill.com:4200',
    "https":'http://web.prod.proxy.cargill.com:4200',
} 

#set salesforce instance: leap_prod, leap_sandbox, salt_prod, salt_sandbox, avenger_prod, avenger_sandbox
#gets credentials from JSON file for the selected SF instance
def sf_grabber(instance, proxies):
    with open("creds.json") as f:
        json_load = json.load(f)['instances']
    
    #returns creds for specific instance
    creds = next((item for item in json_load if item['name'] == instance), None)
    
    #connect and query
    sf_kwargs  = {
        'username': creds['username'],
        'password': creds['pass'],
        'organizationId': creds['org_id'],
        'proxies': proxies
    }
    
    #ads test domain to arguement if "sanbox" / adds token if available for avenger
    if "sandbox" in instance:
        sf_kwargs["domain"] = "test"
    if creds.get("token"):
        sf_kwargs["security_token"] = creds["token"]
    return Salesforce(**sf_kwargs)

#steablish SF connections
salt_sf = sf_grabber('salt_prod', proxies)
leap_sf = sf_grabber('leap_prod', proxies)

In [2]:
#get salt cases
#VALUES HERE FEEL WRONG SHOULD INVESTIGATE
salt_cases_df = salt_sf.query_all("""
SELECT Id,
QN_Number__c,  
Account.Name,
Sales_order_number__c,
CS_Case_Number__c,
Additional_Information__c,
Case_Owner__c,
ClosedDate
FROM Case
WHERE CS_BusinessUnit__c = 'Salt'
AND Reason = 'Complaint'
AND Sales_order_number__c = ''
AND QN_Number__c = ''
""")
salt_cases_df = pd.json_normalize(salt_cases_df["records"])[[
    'Id',
    'QN_Number__c',
    'Sales_order_number__c',
    'CS_Case_Number__c',
    'Additional_Information__c',
    'Case_Owner__c',
    'ClosedDate',
    'Account.Name'
]]
salt_cases_df['CS_Case_Number__c'] = salt_cases_df['CS_Case_Number__c'].astype(int)

#string operation for salt
salt_cases_df["summary_text"] = salt_cases_df.apply(
    lambda row: (
        f"{row['QN_Number__c']} - "
        f"{row['Account.Name']} on order "
        f"{row['Sales_order_number__c']}, has a(n) "
        f"{row['Additional_Information__c']}. The case number is "
        f"{row['CS_Case_Number__c']}. Processed by  "
        f"{row['Case_Owner__c']} "
        f"{pd.Timestamp.today().strftime('%m/%d/%Y')}"
    ),
    axis=1
)

In [3]:
#get leap cases
#VALUES HERE FEEL WRONG SHOULD INVESTIGATE
leap_cases_df = leap_sf.query_all("""
SELECT Id,
Account.Name,
Sales_Order_Number__c,
Product__r.Name,
CaseNumber,
CaseUpdate__c,
Owner.Name,
ClosedDate,
credit_debit_memo_number__c
FROM Case
WHERE Quality_Notification_ID__c  <> ''
AND Status = 'Closed'
""")
leap_cases_df = pd.json_normalize(leap_cases_df["records"])[[
    'Id',
    'Account.Name',
    'Sales_Order_Number__c',
    'Product__r.Name',
    'CaseNumber',
    'CaseUpdate__c',
    'Owner.Name',
    'ClosedDate',
    'credit_debit_memo_number__c'
]]

#string operation for leap
leap_cases_df["summary_text"] = leap_cases_df.apply(
    lambda row: (
        f"{row['Account.Name']} on order  "
        f"{row['Sales_Order_Number__c']} has a complaint case "
        f"{row['CaseNumber']} for "
        f"{row['CaseUpdate__c']} "
        f"{row['Product__r.Name']}. "
        f"{row['credit_debit_memo_number__c']}"
    ),
    axis=1
)